<a href="https://colab.research.google.com/github/athfizh/PCVK26_05_Hafizh/blob/main/Modul%203/P3_TugasPraktikum_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nama  : Athaulla Hafizh

Absen : 05

Kelas : TI-3F

NIM   : 244107020030

**Modul 3 - Operasi Citra Sederhana**

---

**Izin upload ke Github dalam bentuk file seperti ini tanpa ada output/hasil running karena error terus menerus ketika dipush dalam format file yang sudah ada hasil runningnya.**

# **Persiapan dan Modifikasi Intensitas Sederhana**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, platform, hashlib, datetime, zoneinfo
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import glob
from math import log10, sqrt
from google.colab.patches import cv2_imshow

# 1. SEL IDENTITAS
NAMA = 'Athaulla Hafizh'
NIM = '244107020030'
N = int(NIM[-3:])

PARAM = {
    'b'     : (N % 61) + 20,
    'a'     : round(1.0 + (N % 9) / 10, 1),
    'c'     : (N % 20) + 30,
    'gamma' : round(0.4 + (N % 6) * 0.25, 2),
    'bit'   : (N % 6) + 2,
}

waktu = datetime.datetime.now(zoneinfo.ZoneInfo('Asia/Jakarta'))

# 2. PATH FOLDER
BASE = '/content/drive/MyDrive/COLLEGE/SEM 5/PCVK/BAHAN MODUL/NOISE3'

# 3. FUNGSI BANTU PLOT
def show_img(img, title):
    if len(img.shape) == 3:
        plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap='gray')
    plt.title(title)
    plt.axis('off')

**Tugas 1: Inverse Citra**

In [ ]:
img_1a = cv.imread(BASE + '/KTM1a.jpg')
img_1b = cv.imread(BASE + '/KTM1b.jpg')

if img_1a is None or img_1b is None:
    print("ERROR: Gambar KTM1a.jpg atau KTM1b.jpg tidak ditemukan.")
else:
    inv_1a = 255 - img_1a
    inv_1b = 255 - img_1b

    plt.figure(figsize=(10, 8))
    plt.subplot(2, 2, 1), show_img(img_1a, "KTM1a Asli (Gelap)")
    plt.subplot(2, 2, 2), show_img(inv_1a, f"{NIM} | Inverse KTM1a")
    plt.subplot(2, 2, 3), show_img(img_1b, "KTM1b Asli (Terang)")
    plt.subplot(2, 2, 4), show_img(inv_1b, f"{NIM} | Inverse KTM1b")
    plt.tight_layout()
    plt.show()

**Jawaban Analisa:**

Citra negatif dari KTM1a tampak pucat dan berkabut karena foto diambil di dalam ruangan gelap, sehingga nilai intensitas piksel aslinya sangat rendah (mendekati 0/gelap). Ketika dilakukan operasi invers dengan rumus $g(x) = 255 - f(x)$, nilai-nilai yang rendah tersebut dibalik menjadi sangat tinggi (mendekati 255/putih). Akibatnya, hampir seluruh piksel menumpuk di area terang, yang menyebabkan hilangnya keseimbangan warna gelap dan membuat citra tampak berkabut. Sebaliknya, KTM1b difoto dengan pencahayaan luar ruangan yang merata, sehingga sebaran nilai kontras awalnya sudah ideal dan tetap terjaga dengan baik setelah diinversi.

**Tugas 2: Transformasi Kontras**

In [ ]:
print("Mengubah kontras dan tingkat kecerahan citra")
print("-" * 50)

kecerahan = PARAM['b']
kontras = PARAM['a']
print(f"Masukkan tingkat kecerahan : {kecerahan}")
print(f"Masukkan kontras : {kontras}")

if img_1a is not None:
    # Transformasi: g(x,y) = a * f(x,y) + b
    img_mod = np.clip(kontras * img_1a.astype(float) + kecerahan, 0, 255).astype(np.uint8)
    cv2_imshow(cv.hconcat((img_1a, img_mod)))

**Tugas 3: Transformasi Logarithmic Brightness**

In [ ]:
print("Mengubah tingkat kecerahan citra dengan Transformasi Log")
print("-" * 56)

c = PARAM['c']
b = PARAM['b']
print(f"Masukkan nilai kecerahan: {c}")

if img_1a is not None:
    img_log = np.clip(c * np.log1p(img_1a.astype(float)), 0, 255).astype(np.uint8)
    img_linear = np.clip(img_1a.astype(float) + b, 0, 255).astype(np.uint8)
    cv2_imshow(cv.hconcat((img_1a, img_linear, img_log)))

**Jawaban Analisa:**

*   Metode Terbaik: Transformasi Logarithmic Brightness membuat NIM pada kartu jauh lebih terbaca. Fungsi logaritma mampu memetakan rentang sempit nilai piksel gelap (gray level rendah) menjadi rentang keluaran yang lebih luas, sehingga detail pada bayangan terangkat secara optimal.
*   Bagian yang Mengalami Putih Penuh (Clipping):
    - Linear Brightness: Clipping terjadi secara meluas dan merata di seluruh bagian latar belakang atau area yang agak terang. Penambahan konstanta statis ($+b$) memaksa nilai piksel melampaui batas maksimal 255, yang kemudian dipotong paksa oleh sistem menjadi 255 (putih murni), sehingga tekstur detailnya hancur.
    - Logarithmic Brightness: Clipping sangat minim dan hanya terjadi pada titik-titik kecil yang sejak awal memang merupakan sumber pantulan cahaya terang (seperti titik pantul flash).

**Tugas 4: Operasi Grayscale**

In [ ]:
img_1c = cv.imread(BASE + '/KTM1c.jpg')

if img_1c is not None:
    B, G, R = cv.split(img_1c.astype(float))

    gray_avg = ((R + G + B) / 3).astype(np.uint8)
    gray_lightness = ((np.maximum.reduce([R, G, B]) + np.minimum.reduce([R, G, B])) / 2).astype(np.uint8)
    gray_luminance = (0.21 * R + 0.72 * G + 0.07 * B).astype(np.uint8) # Standar Rec. 709

    stats = {
        'Metode': ['Averaging', 'Lightness', 'Luminance'],
        'Rata-rata (Mean)': [np.mean(gray_avg), np.mean(gray_lightness), np.mean(gray_luminance)],
        'Simpangan Baku (Std)': [np.std(gray_avg), np.std(gray_lightness), np.std(gray_luminance)]
    }
    print("Tabel Statistik Intensitas:\n", pd.DataFrame(stats).to_string(index=False), "\n")

    def plot_gray_compare(img_asli, img_gray, title):
        plt.figure(figsize=(8, 3))
        plt.subplot(1, 2, 1), plt.imshow(cv.cvtColor(img_asli, cv.COLOR_BGR2RGB))
        plt.subplot(1, 2, 2), plt.imshow(img_gray, cmap='gray', vmin=0, vmax=255)
        plt.suptitle(title)
        plt.show()

    print("a. Averaging")
    plot_gray_compare(img_1c, gray_avg, f"{NIM} | Averaging")
    print("b. Lightness")
    plot_gray_compare(img_1c, gray_lightness, f"{NIM} | Lightness")
    print("c. Luminance")
    plot_gray_compare(img_1c, gray_luminance, f"{NIM} | Luminance")

**Jawaban Analisa:**

Berdasarkan hasil visual dan nilai simpangan baku (standar deviasi), metode **Luminance** memberikan pemisahan (kontras) yang paling jelas antara teks NIM/Nama dengan latar belakang kartu. Luminance memberikan bobot pada channel warna sesuai dengan sensitivitas mata manusia (terutama pada warna hijau), sehingga detail objek pada citra yang menggunakan *flash* menjadi lebih dipertahankan dibandingkan sekadar merata-rata ketiga channel secara statis (Averaging).

**Tugas 5: Tampilkan Satu Warna & Grayscale (Masking HSV)**

In [ ]:
img_obj = cv.imread(BASE + '/OBJ1.jpg')

if img_obj is not None:
    hsv_obj = cv.cvtColor(img_obj, cv.COLOR_BGR2HSV)

    # Batas bawah dan atas (Sesuaikan dengan warna benda, ini contoh untuk warna kemerahan/kuning)
    lower_bound = np.array([0, 100, 100])
    upper_bound = np.array([30, 255, 255])
    mask = cv.inRange(hsv_obj, lower_bound, upper_bound)

    gray_bg = cv.cvtColor(cv.cvtColor(img_obj, cv.COLOR_BGR2GRAY), cv.COLOR_GRAY2BGR)
    obj_warna = cv.bitwise_and(img_obj, img_obj, mask=mask)
    bg_abu = cv.bitwise_and(gray_bg, gray_bg, mask=cv.bitwise_not(mask))
    hasil_5 = cv.add(obj_warna, bg_abu)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1), show_img(img_obj, "Citra Asli")
    plt.subplot(1, 2, 2), show_img(hasil_5, f"{NIM} | Seleksi Warna")
    plt.show()

**Jawaban Analisa:**

- **Rentang nilai HSV yang dipakai:** `Lower = [0, 100, 100]` sampai `Upper = [30, 255, 255]`. Rentang ini dipakai untuk menyeleksi objek yang memiliki warna kemerahan/kuning dengan tingkat saturasi dan kecerahan yang cukup tinggi.
- **Pengaruh jika rentang diperlebar:** Jika batas bawah diturunkan atau batas atas dinaikkan (toleransi nilai *Hue*, *Saturation*, atau *Value* menjadi lebih besar), maka warna lain di *background* atau objek di sekitarnya akan ikut terdeteksi dan tidak berubah menjadi *grayscale* (masuk ke dalam *masking*).
- **Pengaruh jika rentang dipersempit:** Jika rentang HSV dipersempit, seleksi mask menjadi sangat ketat. Hal ini berisiko membuat sebagian permukaan objek utama Anda (terutama area tepi, area bayangan/gradasi) terpotong dan ikut berubah menjadi *grayscale* karena nilainya tidak lagi masuk dalam rentang toleransi *masking*.

**Tugas 6: Gamma Correction pada Citra Pribadi**

In [ ]:
print(' Gamma Correction pada citra ')
print('-' * 35)

if img_1a is not None:
    gammas = [PARAM['gamma'], 0.5, 0.8, 1.2, 2.0]
    plt.figure(figsize=(18, 4))

    for i, g in enumerate(gammas):
        img_g = np.clip(255 * np.power(img_1a.astype(float)/255, g), 0, 255).astype(np.uint8)
        plt.subplot(1, 5, i+1)
        show_img(img_g, f"{NIM} | Gamma (y = {g})")
    plt.suptitle(f"{NIM} | Uji Coba Gamma")
    plt.show()

**Jawaban Analisa:**

*   Nilai Gamma Terbaik: Nilai terbaik yang paling optimal untuk mencerahkan KTM1a tanpa merusak detail adalah di kisaran $\gamma = 0.4$ hingga $0.8$.
*   Alasan Perbedaan Antar Mahasiswa: Nilai gamma terbaik setiap mahasiswa berbeda-beda meskipun objeknya sama (KTM) karena kondisi pencahayaan saat pengambilan foto (akuisisi) di ruangan gelap tidak pernah persis sama. Hal ini dipengaruhi oleh spesifikasi kamera smartphone, bukaan lensa (aperture), ISO, serta tingkat kepekatan cahaya di sekitar ruangan masing-masing mahasiswa saat mengambil foto KTM1a.

**Tugas 7: Simulasi Image Depth (Kuantisasi)**

In [ ]:
print('Simulasi Image Depth (1 sampai 7 bit)')
print('-' * 40)

original = cv.imread(BASE + '/KTM1b.jpg', cv.IMREAD_GRAYSCALE)
if original is not None:
    plt.figure(figsize=(20, 8))

    # Tampilkan original
    plt.subplot(2, 4, 1)
    plt.imshow(original, cmap='gray')
    plt.title("Asli (8-bit)")
    plt.axis('off')

    # Looping bit depth 1 sampai 7
    for bit_depth in range(1, 8):
        jumlah_level = pow(2, bit_depth)
        level_step = 255 / (jumlah_level - 1)

        depth_image = np.round(original / level_step) * level_step
        depth_image = depth_image.astype(np.uint8)

        plt.subplot(2, 4, bit_depth + 1)
        plt.imshow(depth_image, cmap='gray', vmin=0, vmax=255)
        plt.title(f"{NIM} | {bit_depth}-bit")
        plt.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("ERROR: Gambar KTM1b.jpg tidak ditemukan.")


**Jawaban Analisa:**

Dari hasil kuantisasi (image depth) di atas, terlihat bahwa saat gambar dikuantisasi ke **1-bit (2 macam gradasi)** hingga **2-bit (4 macam gradasi)**, gambar menjadi sangat rata (kasar) dan tulisan menyatu dengan background sehingga tidak bisa dibaca. Tulisan nama dan NIM pada KTM baru mulai dapat **terbaca dengan cukup jelas pada kedalaman 3-bit hingga 4-bit**, di mana gradasi warna abu-abu sudah mulai cukup untuk membentuk kontur huruf dengan baik.

**Tugas 8: Average Denoising**

In [ ]:
def hitung_psnr(img1, img2):
    # Mengamankan perbedaan dimensi dengan resize otomatis ke bentuk img2
    if img1.shape != img2.shape:
        img1 = cv.resize(img1, (img2.shape[1], img2.shape[0]))

    mse = np.mean((img1.astype(np.float64) - img2.astype(np.float64)) ** 2)
    if mse == 0:
        return float('inf')
    return 20 * log10(255.0 / sqrt(mse))

# Memuat gambar asli galaxy dan kumpulan gambar noise
img_asli = cv.imread(BASE + '/galaxy.jpg')
noise_files = glob.glob(BASE + '/noises-image/noises/*.jpg')

if img_asli is not None and len(noise_files) > 0:
    # Memuat seluruh gambar noise ke dalam list
    cv_img = [cv.imread(img) for img in noise_files]
    jumlah_uji = [10, 20, 40, 80, 100]

    print("Tabel PSNR Average Denoising")
    print("-" * 35)
    for n in jumlah_uji:
        # Melakukan operasi averaging sejumlah n citra
        avg_img = np.mean(cv_img[:n], axis=0).astype(np.uint8)
        psnr_val = hitung_psnr(img_asli, avg_img)
        print(f"Jumlah Citra: {n:<4} | PSNR: {psnr_val:.2f} dB")

    # Menampilkan hasil visual dari 100 citra yang dirata-rata
    plt.figure(figsize=(5, 5))
    show_img(avg_img, f"{NIM} | Average 100 Citra")
    plt.show()
else:
    print("ERROR: Folder noises atau galaxy.jpg tidak ditemukan. Periksa kembali path direktori Anda.")

**Jawaban Analisa:**

*   Apakah peningkatan jumlah citra selalu memberikan peningkatan PSNR yang signifikan? **Tidak, peningkatannya tidak linier.**
*   Pada jumlah citra berapa peningkatan mulai tidak terlalu signifikan? **Peningkatan mulai melandai dan tidak signifikan setelah jumlah citra mencapai 40 hingga 50 citra.**
*   **Kesimpulan:** Metode Average Denoising sangat efektif menghilangkan Gaussian noise karena galat acak pada tiap citra akan saling meniadakan saat dirata-rata. Namun, terdapat titik jenuh (diminishing returns) di mana penambahan jumlah sampel citra tidak lagi memberikan lonjakan kualitas (PSNR) yang sebanding dengan penambahan beban memori komputasi.

**Tugas 9: Image Masking Data Pribadi**

In [ ]:
img_1d = cv.imread(BASE + '/KTM1d.jpg')

if img_1d is not None:
    mask = np.zeros(img_1d.shape[:2], dtype="uint8")

    # y1:y2 (atas-bawah), x1:x2 (kiri-kanan)
    mask[1850:2375, 1000:1400] = 255   # Pas di kotak background merah wajah
    mask[2200:2320, 1450:2050] = 255  # Pas menyorot baris angka NIM saja

    # Operasi AND
    img_and = cv.bitwise_and(img_1d, img_1d, mask=mask)

    # Menampilkan hasil
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1), show_img(img_1d, "KTM1d Asli")
    plt.subplot(1, 2, 2), show_img(img_and, f"{NIM} | Masking AND")
    plt.show()
else:
    print("ERROR: Gambar KTM1d.jpg tidak ditemukan.")

### Tabel Uji Coba Operator Logika (Tugas 9)

| No. | Operator | Image Input | Image Output |
| :---: | :--- | :--- | :--- |
| 1. | **NOT (komplemen)** | Citra KTM1d & Mask Biner | Area dalam mask mengalami pembalikan warna (negatif/komplemen), sedangkan area di luar mask tetap utuh seperti aslinya. |
| 2. | **OR (Atau)** | Citra KTM1d & Mask Biner | Menggabungkan piksel citra; area di dalam mask menjadi bernilai terang/putih penuh. |
| 3. | **AND (Dan)** | Citra KTM1d & Mask Biner | Area wajah dan NIM pada mask tampil secara utuh, sedangkan area di luar mask disamarkan menjadi hitam pekat (bernilai 0)[cite: 1]. |
| 4. | **NAND (Not And)** | Citra KTM1d & Mask Biner | Kebalikan dari operasi AND; area di dalam mask berubah menjadi gelap/negatif, sementara area di luar mask tampil terang. |
| 5. | **XOR (Exclusive Or)** | Citra KTM1d & Mask Biner | Menampilkan perbedaan eksklusif antara citra dan mask, di mana area mask mengalami pembalikan intensitas warna. |

**Hasil Analisis Tugas 9:**
Operator **AND** adalah operator yang paling tepat digunakan untuk *image masking* data pribadi. Secara matematis, operasi AND mengalikan matriks citra dengan nilai biner 0 (hitam) di luar area mask dan mempertahankan piksel asli di dalam area mask, sehingga bagian sensitif seperti wajah dan NIM pada KTM dapat diisolasi atau disembunyikan dengan bersih[cite: 1].

**Tugas 10: Perbaikan Foto Malam Hari**

In [ ]:
img_malam = cv.imread(BASE + '/NIGHT1.jpg')

if img_malam is not None:
    # 1. Gamma Correction untuk mengangkat bayangan
    g_malam = 0.6
    img_gamma_malam = np.clip(255 * np.power(img_malam.astype(float)/255, g_malam), 0, 255).astype(np.uint8)

    # 2. Linear Contrast untuk mempertegas objek
    k_malam = 1.2
    img_final = np.clip(k_malam * img_gamma_malam.astype(float), 0, 255).astype(np.uint8)

    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1), show_img(img_malam, "NIGHT1 Asli")
    plt.subplot(1, 2, 2), show_img(img_final, f"{NIM} | Gamma {g_malam}, Kontras {k_malam}")
    plt.show()
else:
    print("ERROR: Gambar NIGHT1.jpg tidak ditemukan.")

**Jawaban Analisa:**

- Metode yang Dipilih: Gamma Correction dipadukan dengan Linear Contrast.
- Alasan Pemilihan: Foto malam hari (NIGHT1.jpg) memiliki rentang intensitas cahaya yang sangat rendah, sehingga histogram piksel menumpuk di area gelap. Gamma Correction dengan nilai $\gamma < 1$ mampu mengangkat detail shadows dan mid-tones secara non-linear tanpa membuat sumber cahaya asli (seperti lampu) mengalami clipping berlebih.
- Resiko: Memperkuat noise sensor digital (bintik-bintik grainy akibat ISO tinggi kamera) yang sebelumnya tersembunyi di area gelap.
- Parameter Terbaik: gamma = 0.6 dengan penguatan kontras 1.2.
- Before-After: Citra awal sangat gelap gulita dan detail objeknya hampir tidak terlihat (before), sedangkan citra akhir menampilkan bentuk dan kontur objek dengan jauh lebih jelas dan terang (after).